# 🏋️ Week 3: Preprocessing & Feature Engineering Practice (Enhanced)

This notebook provides comprehensive practice on data preprocessing with **detailed explanations** of:
- **WHAT** each technique is and when to use it
- **WHY** it works (the math/statistics behind it)
- **HOW** to implement it correctly
- **WHEN** to use each approach in real-world scenarios

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
np.random.seed(42)
print("✅ Imports ready!")

---

## 📚 Exercise 1: Handling Missing Values

### What Are Missing Values?
Missing values (NaN, NULL) occur when data is not available for certain observations. They arise from:
- Data collection errors
- Survey non-responses
- Data integration issues
- Intentionally missing (not applicable)

### Types of Missing Data

| Type | Full Name | Description | Example |
|------|-----------|-------------|---------|
| **MCAR** | Missing Completely at Random | No pattern, truly random | Survey question skipped randomly |
| **MAR** | Missing at Random | Related to other observed variables | Income missing, but correlated with age |
| **MNAR** | Missing Not at Random | Missing depends on the missing value itself | High-income people hide income |

### Imputation Strategies

| Strategy | How It Works | When to Use | Pros | Cons |
|----------|--------------|-------------|------|------|
| **Mean** | Replace with column mean | Numerical, MCAR, symmetric | Simple, fast | Reduces variance, affected by outliers |
| **Median** | Replace with column median | Numerical, skewed data | Robust to outliers | May not be realistic value |
| **Mode** | Replace with most frequent | Categorical | Works for categories | May overrepresent one value |
| **KNN** | Impute from k nearest neighbors | Complex patterns | Uses multiple features | Slow for large datasets |
| **Forward Fill** | Use previous value | Time series | Preserves trends | Assumes continuity |
| **Drop** | Remove rows/columns | >50% missing, MCAR | Simple | Loses data |

### When to Use Each Strategy

```
Missing % < 5%:     → Impute or drop rows
Missing 5-30%:      → Impute (mean/median/KNN)
Missing 30-50%:     → Consider dropping column + imputing
Missing > 50%:      → Drop column (likely uninformative)
```

### Why Median is Often Better Than Mean

```python
data = [10, 12, 14, 15, 200]  # 200 is outlier
mean = 50.2   # Pulled by outlier
median = 14   # Robust to outlier
```

---

In [ ]:
def smart_impute(df: pd.DataFrame, missing_threshold: float = 0.5) -> pd.DataFrame:
    """
    Intelligently impute missing values based on column characteristics.
    
    Algorithm:
    1. For each column, calculate missing percentage
    2. If > threshold: drop column
    3. For numerical columns:
       - Use median if skewed (|skewness| > 1)
       - Use mean if approximately normal
    4. For categorical columns: use mode
    
    Args:
        df: DataFrame with missing values
        missing_threshold: Drop columns with more missing than this
    
    Returns:
        DataFrame with imputed values
    """
    # YOUR CODE HERE
    pass


# Test data with various missing patterns
df_test = pd.DataFrame({
    'age': [25, 30, np.nan, 45, 50, np.nan, 35, 40, 28, 33],
    'income': [50000, np.nan, 75000, 100000, 500000, 60000, np.nan, 80000, 45000, 70000],  # Skewed!
    'city': ['NYC', 'LA', np.nan, 'NYC', 'Chicago', 'LA', 'NYC', np.nan, 'LA', 'Chicago'],
    'mostly_missing': [np.nan, np.nan, np.nan, np.nan, np.nan, 1, np.nan, np.nan, np.nan, np.nan]
})

print("Before imputation:")
print(df_test.isnull().sum())
print(f"\nIncome skewness: {df_test['income'].skew():.2f}")

df_imputed = smart_impute(df_test)
print("\nAfter imputation:")
print(df_imputed.isnull().sum())

In [ ]:
# Solution with detailed logging
def smart_impute_solution(df: pd.DataFrame, missing_threshold: float = 0.5) -> pd.DataFrame:
    df = df.copy()
    cols_dropped = []
    imputation_log = []
    
    for col in df.columns:
        missing_pct = df[col].isnull().sum() / len(df)
        
        # Drop if too much missing
        if missing_pct > missing_threshold:
            cols_dropped.append(col)
            imputation_log.append(f"{col}: DROPPED ({missing_pct:.0%} missing)")
            continue
        
        # Skip if no missing
        if missing_pct == 0:
            continue
        
        # Numerical columns
        if df[col].dtype in ['int64', 'float64']:
            skewness = df[col].skew()
            
            if abs(skewness) > 1:  # Skewed distribution
                fill_value = df[col].median()
                method = f"MEDIAN (skew={skewness:.2f})"
            else:  # Normal-ish distribution
                fill_value = df[col].mean()
                method = f"MEAN (skew={skewness:.2f})"
            
            df[col].fillna(fill_value, inplace=True)
            imputation_log.append(f"{col}: {method} → {fill_value:.2f}")
            
        # Categorical columns
        else:
            fill_value = df[col].mode()[0]
            df[col].fillna(fill_value, inplace=True)
            imputation_log.append(f"{col}: MODE → '{fill_value}'")
    
    # Drop columns
    df.drop(columns=cols_dropped, inplace=True)
    
    # Print log
    print("Imputation Log:")
    print("-" * 50)
    for log in imputation_log:
        print(f"  {log}")
    
    return df

# Run solution
print("\nApplying smart imputation...\n")
df_result = smart_impute_solution(df_test)

print("\nResult:")
print(df_result.head())

---

## 📚 Exercise 2: Handling Outliers

### What Are Outliers?
**Outliers** are data points that differ significantly from other observations. They can be:
- **Legitimate**: Real extreme values (billionaire in income data)
- **Errors**: Data entry mistakes, sensor failures

### Methods to Detect Outliers

#### 1. IQR Method (Tukey's Fences)
```
Q1 = 25th percentile
Q3 = 75th percentile
IQR = Q3 - Q1

Lower Bound = Q1 - 1.5 × IQR
Upper Bound = Q3 + 1.5 × IQR

Outliers: values < Lower or > Upper
```

#### 2. Z-Score Method
$$z = \frac{x - \mu}{\sigma}$$

Values with |z| > 3 are typically considered outliers.

#### 3. Percentile Method
Values below 1st percentile or above 99th percentile.

### Handling Strategies

| Strategy | How It Works | When to Use |
|----------|--------------|-------------|
| **Remove** | Delete outlier rows | Clear errors, small dataset |
| **Cap (Winsorize)** | Limit to bounds | Preserve info, reduce impact |
| **Transform** | Log, sqrt transformation | Heavily skewed data |
| **Keep** | Use robust methods | Legitimate outliers |

### Why Use IQR Over Z-Score?
- **IQR is robust**: Not affected by extreme values
- **Z-score uses mean/std**: Which are themselves affected by outliers

### Real-World Application
- **Finance**: Cap extreme returns to reduce volatility impact
- **Healthcare**: Flag unusual vital signs for review
- **Manufacturing**: Detect faulty sensor readings

---

In [ ]:
def handle_outliers(df: pd.DataFrame, columns: list, method='cap', iqr_multiplier=1.5) -> pd.DataFrame:
    """
    Detect and handle outliers using IQR method.
    
    Algorithm:
    1. For each column, calculate Q1, Q3, IQR
    2. Define bounds: Q1 - 1.5*IQR, Q3 + 1.5*IQR
    3. Either cap values at bounds or remove rows
    
    Args:
        df: Input DataFrame
        columns: Columns to check for outliers
        method: 'cap' (winsorize) or 'remove'
        iqr_multiplier: Multiplier for IQR (1.5 standard, 3.0 for extreme)
    
    Returns:
        DataFrame with handled outliers
    """
    # YOUR CODE HERE
    pass


# Test with outliers
df_outliers = pd.DataFrame({
    'value': [10, 12, 14, 15, 16, 100, 11, 13, -50, 15, 14, 12, 18, 200],
    'normal': np.random.normal(50, 10, 14)
})

print("Original data:")
print(f"  Min: {df_outliers['value'].min()}, Max: {df_outliers['value'].max()}")
print(f"  Mean: {df_outliers['value'].mean():.2f}")

df_capped = handle_outliers(df_outliers.copy(), ['value'], method='cap')
print(f"\nAfter capping:")
print(f"  Min: {df_capped['value'].min():.2f}, Max: {df_capped['value'].max():.2f}")
print(f"  Mean: {df_capped['value'].mean():.2f}")

In [ ]:
# Solution with visualization
def handle_outliers_solution(df: pd.DataFrame, columns: list, method='cap', iqr_multiplier=1.5) -> pd.DataFrame:
    df = df.copy()
    outlier_report = []
    
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - iqr_multiplier * IQR
        upper_bound = Q3 + iqr_multiplier * IQR
        
        # Count outliers
        n_lower = (df[col] < lower_bound).sum()
        n_upper = (df[col] > upper_bound).sum()
        
        outlier_report.append({
            'column': col,
            'Q1': Q1,
            'Q3': Q3,
            'IQR': IQR,
            'lower_bound': lower_bound,
            'upper_bound': upper_bound,
            'n_lower_outliers': n_lower,
            'n_upper_outliers': n_upper
        })
        
        if method == 'cap':
            df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)
        elif method == 'remove':
            df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]
    
    # Print report
    print("Outlier Detection Report:")
    print("=" * 60)
    for report in outlier_report:
        print(f"\nColumn: {report['column']}")
        print(f"  Q1: {report['Q1']:.2f}, Q3: {report['Q3']:.2f}, IQR: {report['IQR']:.2f}")
        print(f"  Bounds: [{report['lower_bound']:.2f}, {report['upper_bound']:.2f}]")
        print(f"  Outliers: {report['n_lower_outliers']} below, {report['n_upper_outliers']} above")
    
    return df

# Compare methods
print("=== CAPPING ===")
df_capped = handle_outliers_solution(df_outliers.copy(), ['value'], method='cap')

print("\n=== REMOVAL ===")
df_removed = handle_outliers_solution(df_outliers.copy(), ['value'], method='remove')
print(f"\nRows after removal: {len(df_removed)} (was {len(df_outliers)})")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].boxplot(df_outliers['value'])
axes[0].set_title('Original (with outliers)')

axes[1].boxplot(df_capped['value'])
axes[1].set_title('After Capping')

axes[2].boxplot(df_removed['value'])
axes[2].set_title('After Removal')

plt.tight_layout()
plt.show()

---

## 📚 Exercise 3: Feature Scaling

### Why Scale Features?
Many ML algorithms are sensitive to feature scales:
- **Distance-based**: KNN, SVM, K-Means (features with larger ranges dominate)
- **Gradient-based**: Neural networks, linear regression (faster convergence)
- **Tree-based**: Generally NOT affected (split on value thresholds)

### Scaling Methods

#### 1. StandardScaler (Z-score Normalization)
$$x_{\text{scaled}} = \frac{x - \mu}{\sigma}$$

- **Result**: Mean = 0, Std = 1
- **Best for**: Normally distributed data, when outliers exist
- **Use with**: SVM, Linear Regression, Neural Networks

#### 2. MinMaxScaler
$$x_{\text{scaled}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

- **Result**: Values in [0, 1]
- **Best for**: When you need bounded range, image data
- **Caution**: Sensitive to outliers!

#### 3. RobustScaler
$$x_{\text{scaled}} = \frac{x - \text{median}}{\text{IQR}}$$

- **Result**: Centered at 0, scaled by IQR
- **Best for**: Data with outliers
- **Why robust**: Median and IQR aren't affected by outliers

### Comparison Table

| Scaler | Robust to Outliers | Output Range | Use Case |
|--------|-------------------|--------------|----------|
| StandardScaler | No | Unbounded | General purpose |
| MinMaxScaler | No | [0, 1] | Neural networks, images |
| RobustScaler | Yes | Unbounded | Data with outliers |
| MaxAbsScaler | No | [-1, 1] | Sparse data |

### Critical Rule: Fit on Train, Transform Both
```python
# CORRECT:
scaler.fit(X_train)           # Learn params from train only!
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)  # Use train's params

# WRONG (data leakage!):
scaler.fit(X)  # Uses test data information
```

---

In [ ]:
def compare_scalers(df: pd.DataFrame, columns: list):
    """
    Compare different scaling methods on given columns.
    
    Scalers to compare:
    - StandardScaler
    - MinMaxScaler  
    - RobustScaler
    
    Args:
        df: Input DataFrame
        columns: Columns to scale
    
    Returns:
        Dict of {scaler_name: scaled_DataFrame}
    """
    # YOUR CODE HERE
    pass


# Test data with different distributions
np.random.seed(42)
df_scale = pd.DataFrame({
    'normal': np.random.normal(100, 15, 100),      # Normal distribution
    'skewed': np.random.exponential(50, 100),       # Skewed with potential outliers
    'with_outliers': np.append(np.random.normal(50, 10, 95), [200, 300, 400, 500, 600])  # Outliers
})

results = compare_scalers(df_scale, ['normal', 'skewed', 'with_outliers'])
for name, df in results.items():
    print(f"\n{name}:")
    print(df.describe().round(2))

In [ ]:
# Solution with visualization
def compare_scalers_solution(df: pd.DataFrame, columns: list):
    scalers = {
        'StandardScaler': StandardScaler(),
        'MinMaxScaler': MinMaxScaler(),
        'RobustScaler': RobustScaler()
    }
    
    results = {}
    
    for name, scaler in scalers.items():
        scaled_data = scaler.fit_transform(df[columns])
        results[name] = pd.DataFrame(scaled_data, columns=columns)
    
    return results

# Get results
results = compare_scalers_solution(df_scale, ['normal', 'skewed', 'with_outliers'])

# Visualize all scalers for 'with_outliers' column
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Original
axes[0, 0].hist(df_scale['with_outliers'], bins=20, edgecolor='black')
axes[0, 0].set_title(f"Original\nMean={df_scale['with_outliers'].mean():.1f}, Std={df_scale['with_outliers'].std():.1f}")
axes[0, 0].axvline(df_scale['with_outliers'].mean(), color='red', linestyle='--', label='Mean')

# StandardScaler
data = results['StandardScaler']['with_outliers']
axes[0, 1].hist(data, bins=20, edgecolor='black', color='blue', alpha=0.7)
axes[0, 1].set_title(f"StandardScaler\nMean={data.mean():.2f}, Std={data.std():.2f}")
axes[0, 1].axvline(0, color='red', linestyle='--', label='Mean=0')

# MinMaxScaler
data = results['MinMaxScaler']['with_outliers']
axes[1, 0].hist(data, bins=20, edgecolor='black', color='green', alpha=0.7)
axes[1, 0].set_title(f"MinMaxScaler\nMin={data.min():.2f}, Max={data.max():.2f}")

# RobustScaler
data = results['RobustScaler']['with_outliers']
axes[1, 1].hist(data, bins=20, edgecolor='black', color='orange', alpha=0.7)
axes[1, 1].set_title(f"RobustScaler\nMedian={data.median():.2f}, IQR-based")
axes[1, 1].axvline(0, color='red', linestyle='--', label='Median=0')

plt.tight_layout()
plt.show()

# Summary comparison
print("\nScaler Comparison for 'with_outliers' column:")
print("=" * 60)
print(f"{'Scaler':<20} {'Min':<10} {'Max':<10} {'Mean':<10} {'Std':<10}")
print("-" * 60)
print(f"{'Original':<20} {df_scale['with_outliers'].min():<10.2f} {df_scale['with_outliers'].max():<10.2f} {df_scale['with_outliers'].mean():<10.2f} {df_scale['with_outliers'].std():<10.2f}")
for name, df in results.items():
    col = df['with_outliers']
    print(f"{name:<20} {col.min():<10.2f} {col.max():<10.2f} {col.mean():<10.2f} {col.std():<10.2f}")

---

## 📚 Exercise 4: Categorical Encoding

### Why Encode Categories?
ML algorithms work with numbers, not strings. We need to convert categorical data to numerical form.

### Types of Categorical Variables

| Type | Has Order? | Example | Encoding |
|------|------------|---------|----------|
| **Nominal** | No | Color (red, blue, green) | One-Hot |
| **Ordinal** | Yes | Size (S, M, L, XL) | Label/Ordinal |
| **Binary** | No | Gender (M, F) | Label (0, 1) |

### Encoding Methods

#### 1. One-Hot Encoding (Dummy Variables)
```
Color    →    is_red  is_blue  is_green
red           1       0        0
blue          0       1        0
green         0       0        1
```

- **Pros**: No ordinal relationship implied
- **Cons**: High cardinality = many columns (curse of dimensionality)
- **Use when**: < 10-15 unique values, nominal data

#### 2. Label Encoding
```
Size    →    encoded
S            0
M            1
L            2
XL           3
```

- **Pros**: Single column, preserves order
- **Cons**: Implies ordinal relationship (may confuse model)
- **Use when**: Ordinal data, tree-based models

#### 3. Target Encoding (Mean Encoding)
Replace category with mean of target variable.

```
City         Target Mean    →    encoded
NYC          0.75               0.75
LA           0.60               0.60
```

- **Pros**: Single column, captures target relationship
- **Cons**: Risk of data leakage, overfitting
- **Use when**: High cardinality, with proper CV

### Cardinality Guidelines

```
Unique Values < 10:     → One-Hot Encoding
Unique Values 10-100:   → Target Encoding or Embedding
Unique Values > 100:    → Feature hashing, embeddings
```

### Drop First Column?
For linear models, drop one column to avoid multicollinearity:
```python
pd.get_dummies(df, drop_first=True)
```

---

In [ ]:
def smart_encode(df: pd.DataFrame, categorical_columns: list, target_col: str = None, cardinality_threshold: int = 10):
    """
    Intelligently encode categorical columns based on cardinality.
    
    Algorithm:
    - Low cardinality (< threshold): One-Hot encode
    - High cardinality (>= threshold): Target encode if target provided, else Label encode
    
    Args:
        df: Input DataFrame
        categorical_columns: Columns to encode
        target_col: Target column for target encoding (optional)
        cardinality_threshold: Threshold for encoding decision
    
    Returns:
        Encoded DataFrame
    """
    # YOUR CODE HERE
    pass


# Test
df_cat = pd.DataFrame({
    'color': ['red', 'blue', 'green', 'red', 'blue', 'green', 'red', 'blue'],  # Low cardinality
    'city': ['NYC', 'LA', 'Chicago', 'Miami', 'Seattle', 'Boston', 'Denver', 'Austin'],  # Higher cardinality
    'target': [1, 0, 1, 1, 0, 0, 1, 0]
})

print("Original:")
print(df_cat)

df_encoded = smart_encode(df_cat, ['color', 'city'], target_col='target', cardinality_threshold=5)
print("\nEncoded:")
print(df_encoded)

In [ ]:
# Solution with detailed logging
def smart_encode_solution(df: pd.DataFrame, categorical_columns: list, target_col: str = None, cardinality_threshold: int = 10):
    df = df.copy()
    encoding_log = []
    
    for col in categorical_columns:
        cardinality = df[col].nunique()
        
        if cardinality < cardinality_threshold:
            # One-Hot Encoding
            dummies = pd.get_dummies(df[col], prefix=col, drop_first=True)
            df = pd.concat([df, dummies], axis=1)
            df.drop(columns=[col], inplace=True)
            encoding_log.append(f"{col}: ONE-HOT (cardinality={cardinality}) → {list(dummies.columns)}")
        else:
            if target_col and target_col in df.columns:
                # Target Encoding
                target_means = df.groupby(col)[target_col].mean()
                df[col + '_encoded'] = df[col].map(target_means)
                encoding_log.append(f"{col}: TARGET ENCODED (cardinality={cardinality})")
            else:
                # Label Encoding
                le = LabelEncoder()
                df[col + '_encoded'] = le.fit_transform(df[col])
                encoding_log.append(f"{col}: LABEL ENCODED (cardinality={cardinality})")
            df.drop(columns=[col], inplace=True)
    
    # Print log
    print("Encoding Log:")
    print("-" * 60)
    for log in encoding_log:
        print(f"  {log}")
    
    return df

# Run
print("\nApplying smart encoding...\n")
df_result = smart_encode_solution(df_cat.copy(), ['color', 'city'], target_col='target', cardinality_threshold=5)
print("\nResult:")
print(df_result)

# Show target encoding in detail
print("\nTarget Encoding Details for 'city':")
target_means = df_cat.groupby('city')['target'].mean()
for city, mean in target_means.items():
    print(f"  {city}: {mean:.2f}")

---

## 📚 Exercise 5: Feature Engineering - Creating New Features

### What is Feature Engineering?
**Feature engineering** is the process of creating new features from existing data to improve model performance. It's often the most impactful part of ML.

### Common Feature Engineering Techniques

#### 1. Mathematical Transformations
```python
# Log transform (for skewed data)
df['log_income'] = np.log1p(df['income'])

# Square root
df['sqrt_area'] = np.sqrt(df['area'])

# Polynomial features
df['income_squared'] = df['income'] ** 2
```

#### 2. Ratio Features
```python
df['debt_to_income'] = df['debt'] / df['income']
df['revenue_per_employee'] = df['revenue'] / df['employees']
```

#### 3. Date/Time Features
```python
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
df['quarter'] = df['date'].dt.quarter
```

#### 4. Binning (Discretization)
```python
# Equal-width bins
df['age_group'] = pd.cut(df['age'], bins=[0, 18, 35, 55, 100], labels=['child', 'young', 'middle', 'senior'])

# Equal-frequency bins (quantiles)
df['income_quartile'] = pd.qcut(df['income'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
```

#### 5. Aggregation Features
```python
# Customer-level aggregations
df['avg_order_value'] = df.groupby('customer_id')['order_value'].transform('mean')
df['total_orders'] = df.groupby('customer_id')['order_id'].transform('count')
```

### Why Feature Engineering Matters

| Raw Feature | Engineered Feature | Improvement |
|-------------|-------------------|-------------|
| timestamp | hour_of_day, is_business_hours | Captures patterns |
| latitude, longitude | distance_to_center | Reduces dimensionality |
| income, debt | debt_to_income_ratio | Domain knowledge |

---

In [ ]:
def extract_date_features(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    """
    Extract comprehensive features from a datetime column.
    
    Features to create:
    - year, month, day, day_of_week, day_of_year
    - is_weekend, is_month_start, is_month_end
    - quarter, week_of_year
    - hour (if timestamp), is_business_hours
    
    Args:
        df: Input DataFrame
        date_col: Name of datetime column
    
    Returns:
        DataFrame with new date features
    """
    # YOUR CODE HERE
    pass


# Test
df_dates = pd.DataFrame({
    'transaction_date': pd.date_range('2024-01-01 09:00:00', periods=10, freq='3H'),
    'amount': np.random.randint(10, 1000, 10)
})

print("Original:")
print(df_dates)

df_with_features = extract_date_features(df_dates, 'transaction_date')
print("\nWith date features:")
print(df_with_features)

In [ ]:
# Solution with comprehensive feature extraction
def extract_date_features_solution(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    df = df.copy()
    dt = df[date_col]
    
    # Ensure datetime type
    if not pd.api.types.is_datetime64_any_dtype(dt):
        dt = pd.to_datetime(dt)
    
    # Basic date components
    df['year'] = dt.dt.year
    df['month'] = dt.dt.month
    df['day'] = dt.dt.day
    df['day_of_week'] = dt.dt.dayofweek  # 0=Monday, 6=Sunday
    df['day_of_year'] = dt.dt.dayofyear
    df['week_of_year'] = dt.dt.isocalendar().week
    df['quarter'] = dt.dt.quarter
    
    # Binary features
    df['is_weekend'] = (dt.dt.dayofweek >= 5).astype(int)
    df['is_month_start'] = dt.dt.is_month_start.astype(int)
    df['is_month_end'] = dt.dt.is_month_end.astype(int)
    
    # Time-based (if timestamp has time component)
    if dt.dt.hour.sum() > 0:  # Check if hours are present
        df['hour'] = dt.dt.hour
        df['is_business_hours'] = ((dt.dt.hour >= 9) & (dt.dt.hour < 17)).astype(int)
        df['time_of_day'] = pd.cut(
            dt.dt.hour, 
            bins=[0, 6, 12, 18, 24], 
            labels=['night', 'morning', 'afternoon', 'evening'],
            include_lowest=True
        )
    
    return df

# Run
df_result = extract_date_features_solution(df_dates, 'transaction_date')
print("Extracted Features:")
print(df_result.columns.tolist())
print("\n")
print(df_result.head(10))

---

## 📚 Exercise 6: Building Preprocessing Pipelines

### Why Use Pipelines?
1. **Reproducibility**: Same transformations applied consistently
2. **Prevent Data Leakage**: Fit only on training data
3. **Cleaner Code**: Chain transformations
4. **Easy Deployment**: Single object to save/load

### Pipeline Components

#### sklearn Pipeline
```python
from sklearn.pipeline import Pipeline

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
```

#### ColumnTransformer (for mixed data types)
```python
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_columns),
    ('cat', categorical_pipeline, categorical_columns)
])
```

### Common Pipeline Pattern
```python
# Complete ML Pipeline
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier())
])

# Fit and predict
full_pipeline.fit(X_train, y_train)
predictions = full_pipeline.predict(X_test)
```

---

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score

def build_preprocessing_pipeline(numeric_cols: list, categorical_cols: list):
    """
    Build a complete preprocessing pipeline for mixed data.
    
    Pipeline structure:
    - Numeric: Impute (median) → Scale (standard)
    - Categorical: Impute (most frequent) → One-Hot encode
    
    Args:
        numeric_cols: List of numeric column names
        categorical_cols: List of categorical column names
    
    Returns:
        ColumnTransformer preprocessing pipeline
    """
    # YOUR CODE HERE
    pass

In [ ]:
# Solution with full ML pipeline
def build_preprocessing_pipeline_solution(numeric_cols: list, categorical_cols: list):
    # Numeric pipeline
    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    
    # Categorical pipeline
    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
    ])
    
    # Combine with ColumnTransformer
    preprocessor = ColumnTransformer([
        ('num', numeric_pipeline, numeric_cols),
        ('cat', categorical_pipeline, categorical_cols)
    ])
    
    return preprocessor

# Create sample data with mixed types
np.random.seed(42)
n_samples = 200

df_mixed = pd.DataFrame({
    'age': np.random.randint(18, 70, n_samples).astype(float),
    'income': np.random.normal(50000, 15000, n_samples),
    'education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n_samples),
    'city': np.random.choice(['NYC', 'LA', 'Chicago', 'Miami'], n_samples),
    'target': np.random.randint(0, 2, n_samples)
})

# Add some missing values
df_mixed.loc[np.random.choice(n_samples, 20), 'age'] = np.nan
df_mixed.loc[np.random.choice(n_samples, 15), 'income'] = np.nan
df_mixed.loc[np.random.choice(n_samples, 10), 'education'] = np.nan

# Define columns
numeric_cols = ['age', 'income']
categorical_cols = ['education', 'city']

# Build complete pipeline with model
preprocessor = build_preprocessing_pipeline_solution(numeric_cols, categorical_cols)

full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Split data
X = df_mixed.drop('target', axis=1)
y = df_mixed['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Data Summary:")
print(f"  Training samples: {len(X_train)}")
print(f"  Test samples: {len(X_test)}")
print(f"  Missing values in train: {X_train.isnull().sum().sum()}")

# Cross-validation
cv_scores = cross_val_score(full_pipeline, X_train, y_train, cv=5, scoring='accuracy')
print(f"\nCross-Validation Scores: {cv_scores.round(3)}")
print(f"Mean CV Score: {cv_scores.mean():.3f} (+/- {cv_scores.std()*2:.3f})")

# Fit and evaluate
full_pipeline.fit(X_train, y_train)
test_score = full_pipeline.score(X_test, y_test)
print(f"\nTest Score: {test_score:.3f}")

# Show transformed feature names
print("\nPipeline Structure:")
print(full_pipeline)

---

## 📋 Week 3 Practice Summary

### Concepts Covered

| Topic | Key Technique | When to Use |
|-------|--------------|-------------|
| Missing Values | Median/Mean/Mode imputation | Based on data type and distribution |
| Outliers | IQR-based capping/removal | Numerical features with extreme values |
| Scaling | StandardScaler, RobustScaler | Distance/gradient-based models |
| Encoding | One-Hot vs Target encoding | Based on cardinality |
| Feature Engineering | Date extraction, ratios | Domain-specific improvements |
| Pipelines | ColumnTransformer | Production ML systems |

### Exercises Completed
- [ ] Smart Missing Value Imputation
- [ ] Outlier Detection and Handling
- [ ] Scaler Comparison
- [ ] Categorical Encoding Strategies
- [ ] Date Feature Extraction
- [ ] Complete Preprocessing Pipeline

### Interview Tips
1. **Always fit on train data only** - Prevent data leakage
2. **Know when to use each scaler** - StandardScaler vs RobustScaler vs MinMaxScaler
3. **Consider cardinality** - One-hot for low, target encoding for high
4. **Use pipelines** - Shows production-ready thinking
5. **Domain knowledge** - Best features come from understanding the problem

---
**Ready for Week 4: End-to-End Project!** 🚀